# 🚀 Project MIP — Institutional Quantitative Trading Desk & Momentum Platform
### Standalone Mobile & Cloud Engine • Python 3.10+ • C99 SIMD Acceleration

Welcome to **Project MIP (Momentum Investing Platform)** on Google Colab.
This workstation is engineered for high-performance quantitative research, multi-universe momentum scanning across all 36 official NSE indices, and blazing-fast brute force backtesting.

---

## 🛠️ Cell 1: Environment Setup, Drive Mount & Fast NVMe Database Staging
Initializes the workstation, mounts Google Drive, and stages the database to local NVMe storage to circumvent cloud latency.

> **Storage Quota & High-Compression Note:** To prevent filling your Google Drive storage quota, `universe.db` is stored permanently in your Drive as a highly compressed gzip archive (`universe.db.gz`, ~100 MB saving over 70% storage space). When this cell runs, it streams and decompresses the archive to local high-speed NVMe storage in ~1.5s for ultra-fast <1ms queries. Any new market data fetched during scans is automatically compressed and saved back to your Google Drive.


In [ ]:
#@title 🛠️ Cell 1: Environment Setup, Drive Mount & Fast NVMe Database Staging { display-mode: "form" }
# ==============================================================================
# CELL 1: Environment Setup, Drive Mount & Fast NVMe Database Staging
# ==============================================================================
import os, sys, platform, shutil, inspect, importlib, time
from pathlib import Path

print("🔍 PROBING CLOUD COMPUTE ENVIRONMENT...")
print(f"  • Platform:     {platform.platform()}")
print(f"  • Python:       {sys.version.split()[0]}")
print(f"  • Architecture: {platform.machine()}")
print(f"  • CPU Cores:    {os.cpu_count()}")

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
WORKSPACE_DIR = Path("/content/Project_MIP_Colab") if IN_COLAB else Path.cwd()

if IN_COLAB and (WORKSPACE_DIR / ".git").exists():
    print("\n🔄 Fetching latest updates from GitHub repository...")
    !git -C {WORKSPACE_DIR} pull origin main
elif IN_COLAB and not (WORKSPACE_DIR / "core").exists():
    print("\n📦 Cloning repository into Colab workspace...")
    !git clone https://github.com/Afylx01/Project_MIP_Colab.git {WORKSPACE_DIR}

if WORKSPACE_DIR.exists():
    os.chdir(WORKSPACE_DIR)
    if str(WORKSPACE_DIR) not in sys.path:
        sys.path.insert(0, str(WORKSPACE_DIR))

# Purge any stale in-memory cached modules from previous cell runs or pre-pull state
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("scripts", "core", "brute_force", "nextgen")):
        sys.modules.pop(mod_name, None)

!pip install -q -r requirements.txt
print(f"\n✓ Workspace active at: {WORKSPACE_DIR}")

# Automatically mount Google Drive if running in Colab
if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    print("\n📁 Mounting Google Drive...")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount notice: {e}")

# Bulletproof module loading with reload & inline fallback
try:
    import scripts.cloud_sync
    importlib.reload(scripts.cloud_sync)
    from scripts.cloud_sync import (
        sync_universe_from_gdrive,
        benchmark_storage_latency,
        is_gzipped_file,
        decompress_file,
        PUBLIC_UNIVERSE_DB_FILE_ID,
        PUBLIC_UNIVERSE_DB_URL,
    )
except Exception:
    for mod_name in list(sys.modules.keys()):
        if mod_name.startswith(("scripts", "core", "brute_force", "nextgen")):
            sys.modules.pop(mod_name, None)
    from scripts.cloud_sync import (
        sync_universe_from_gdrive,
        benchmark_storage_latency,
        is_gzipped_file,
        decompress_file,
        PUBLIC_UNIVERSE_DB_FILE_ID,
        PUBLIC_UNIVERSE_DB_URL,
    )

if 'is_gzipped_file' not in globals() or not callable(globals().get('is_gzipped_file')):
    def is_gzipped_file(file_path):
        p = Path(file_path)
        if not p.exists() or p.stat().st_size < 2:
            return False
        try:
            with open(p, "rb") as f:
                return f.read(2) == b"\x1f\x8b"
        except Exception:
            return False

if 'decompress_file' not in globals() or not callable(globals().get('decompress_file')):
    def decompress_file(src_gz, dest_raw):
        import gzip
        src, dest = Path(src_gz), Path(dest_raw)
        tmp = dest.parent / f"{dest.name}.tmp_decomp_{int(time.time())}"
        with gzip.open(src, "rb") as f_in, open(tmp, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        tmp.replace(dest)
        return src.stat().st_size, dest.stat().st_size

from core.data_engine import get_db_path, get_universe_statistics

print("\n⚡ Staging universe.db to local NVMe storage (with gzip decompress)...")
ok, status, meta = sync_universe_from_gdrive(verbose=True)

db_target = get_db_path()
if not db_target.exists() and IN_COLAB:
    print(f"\n📥 Downloading universe.db directly via Google Drive File ID ({PUBLIC_UNIVERSE_DB_FILE_ID})...")
    !gdown --id {PUBLIC_UNIVERSE_DB_FILE_ID} -O {db_target}
    if db_target.exists() and is_gzipped_file(db_target):
        print("  • Decompressing downloaded gzip archive to SQLite...")
        decompress_file(db_target, db_target)

if db_target.exists():
    stats = get_universe_statistics()
    print("")
    print("✓ DATABASE INTEGRITY CONFIRMED:")
    print(f"  • Total Rows:     {stats['total_rows']:,}")
    print(f"  • Unique Symbols: {stats['unique_symbols']}")
    print(f"  • Date Span:      {stats['min_date']} → {stats['max_date']}")
    print(f"  • Local NVMe Size:{stats['file_size_mb']:.2f} MB")
    print("")
    print("⚡ Benchmarking Storage Latency...")
    bm = benchmark_storage_latency(db_target, num_queries=20)
    print(f"  • Average Latency: {bm['avg_latency_ms']:.2f} ms ({'NVMe Ultra-Fast' if bm['is_nvme_grade'] else 'Network-Bound'})")
else:
    print("")
    print("⚠ universe.db not found on Google Drive.")
    print("ℹ Direct Download Link:")
    print(f"  {PUBLIC_UNIVERSE_DB_URL}")
    print("  Or run: !gdown --id 19LaJGbeVx2lx9RytT0vig0Srqc6G4Q65 -O data/universe.db")

## 📊 Cell 2: Live Momentum Screener Cockpit
**Interactive Colab Form:** Pre-select your target universe, portfolio size, and export options in the **Form Header** at the top of the cell, then click the **Run Cell (▶)** button!

Guarantees scan execution on the **latest actual completed trading day of EOD**, with real-time Telegram dispatch, institutional 14-sheet Excel generation, browser auto-download, and database synchronization audits.

In [ ]:
#@title 📊 Cell 2: Live Momentum Screener Cockpit { display-mode: "form" }
#@markdown ---
#@markdown #### 🎯 Target Universe & Portfolio Size
universe = "NIFTY750" #@param ["NIFTY750", "NIFTY500", "NIFTY50", "NIFTYBANK", "NIFTYIT", "NIFTYAUTO", "NIFTYPHARMA", "ALL"]
top_n = 20 #@param {type:"slider", min:5, max:50, step:5}

#@markdown #### 🔬 Institutional Next-Gen Alpha Overlays (Optional)
use_residual_momentum = True #@param {type:"boolean"}
use_vol_targeting = False #@param {type:"boolean"}
use_delivery_veto = False #@param {type:"boolean"}

#@markdown #### ⚙️ Execution & Export Flags
allow_stale_data = False #@param {type:"boolean"}
generate_excel = True #@param {type:"boolean"}
send_telegram = True #@param {type:"boolean"}
auto_download_excel = True #@param {type:"boolean"}
#@markdown ---
import sys, os, importlib, inspect, io, base64
from pathlib import Path
try:
    from IPython.display import display, HTML
except ImportError:
    display = lambda x: None
    HTML = lambda x: x

# Reload modules to pick up any disk changes
for mod_name in ["core.scanner", "scripts.scanner_hardening", "core.telegram_sender", "core.auto_fetch", "core.settings"]:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

import matplotlib.pyplot as plt
from core.scanner import format_screener_tearsheet
from scripts.scanner_hardening import HardenedScanner, NIFTY_INDICES_CATALOG, trigger_colab_download
from core.data_engine import get_expected_latest_eod, get_latest_date, get_db_path

db_target = get_db_path()
if not db_target.exists():
    raise FileNotFoundError("❌ data/universe.db not found! Please run Cell 1 first to stage the database to local NVMe.")

expected_session = get_expected_latest_eod()
db_latest = get_latest_date()

workspace_dir = Path("/content/Project_MIP_Colab") if Path("/content/Project_MIP_Colab").exists() else Path.cwd()
scanner = HardenedScanner(workspace_dir)

sig = inspect.signature(scanner.run_scan)
scan_kwargs = {
    "as_of_date": None,
    "universe_mode": universe,
    "top_n": top_n,
    "auto_update_universe": True,
    "export_csv": True,
    "export_excel": generate_excel,
    "allow_stale_data": allow_stale_data,
}
if "send_telegram" in sig.parameters:
    scan_kwargs["send_telegram"] = send_telegram
if "auto_download_excel" in sig.parameters:
    scan_kwargs["auto_download_excel"] = False
if "use_vol_targeting" in sig.parameters:
    scan_kwargs["use_vol_targeting"] = use_vol_targeting
if "use_delivery_veto" in sig.parameters:
    scan_kwargs["use_delivery_veto"] = use_delivery_veto
if "use_residual_momentum" in sig.parameters:
    scan_kwargs["use_residual_momentum"] = use_residual_momentum

print(f"⏳ Executing momentum scan on {universe} (Top {top_n})...")
top_df, breadth, sectors, regime, top_etf, deliv = scanner.run_scan(**scan_kwargs)
scanned_date = top_df.attrs.get("scanned_date") or db_latest

# Build tearsheet text
tearsheet_text = format_screener_tearsheet(top_df, breadth, sectors, regime, scanned_date)
# Build Next-Gen Alpha Overlays text
overlay_text = ""
if use_residual_momentum or use_vol_targeting or use_delivery_veto:
    overlay_text = "\n" + "=" * 76 + "\n 🛡️ NEXT-GEN QUANTITATIVE ALPHA OVERLAYS:\n"
    if use_residual_momentum:
        overlay_text += "   • Idiosyncratic Residual Momentum (Blitz et al.): Active (40% Canonical Volar + 60% Idiosyncratic Alpha, 21d Reversal Excluded)\n"
    if use_delivery_veto:
        vetoed = top_df.attrs.get("delivery_vetoed_symbols", [])
        if vetoed:
            overlay_text += f"   • Microstructure Delivery Veto: Filtered {len(vetoed)} distribution candidate(s): {', '.join(vetoed)}\n"
        else:
            overlay_text += "   • Microstructure Delivery Veto: Active (Zero institutional distribution traps detected)\n"
    if use_vol_targeting:
        shield = top_df.attrs.get("shield_scalar", 1.0)
        r_vol = top_df.attrs.get("market_realized_vol", 0.15)
        overlay_text += f"   • Dynamic Volatility Cash Shield: Exposure scaled to {shield*100:.1f}% Equity / {(1-shield)*100:.1f}% Liquid Cash\n     (Market Realized Vol: {r_vol*100:.1f}% vs Target: 12.0%)\n"
    overlay_text += "=" * 76 + "\n"

# Build database audit text
db_audit = top_df.attrs.get("db_audit")
audit_text = ""
if db_audit:
    if db_audit.get("is_updated"):
        audit_text = (
            "\n" + "=" * 76 + "\n"
            "💾 DATABASE AUDIT: 🟢 UPDATED WITH NEW MARKET SESSIONS!\n"
            f"  • New Rows Added:  +{db_audit.get('new_rows_added', 0):,} rows committed to SQLite\n"
            f"  • Date Transition: {db_audit.get('pre_latest_date')} ➔ {db_audit.get('post_latest_date')}\n"
            f"  • Total Rows:      {db_audit.get('post_row_count', 0):,} rows in prices table\n"
            + ("  • Google Drive:    ✓ Backed up to /MyDrive/MIP/universe.db.gz\n" if db_audit.get("gdrive_backed_up") else "")
            + "=" * 76
        )
    else:
        audit_text = (
            "\n" + "=" * 76 + "\n"
            "💾 DATABASE AUDIT: 🔵 ALREADY UP-TO-DATE (No new sessions available)\n"
            f"  • Latest Session:  {db_audit.get('post_latest_date')}\n"
            f"  • Total Rows:      {db_audit.get('post_row_count', 0):,} rows in prices table\n"
            + "=" * 76
        )

# Render Plot to base64 image (Dynamic sizing, prevent figure leak warning)
chart_html = ""
plt.close("all")
if top_df is not None and not top_df.empty and "volar_score" in top_df.columns:
    fig = plt.figure(figsize=(10, 4.5), dpi=100)
    plot_data = top_df.head(10).sort_values("volar_score", ascending=True)
    plt.barh(plot_data["symbol"], plot_data["volar_score"], color="#00adb5")
    plt.title(f"Top 10 Momentum Candidates [{universe}] ({scanned_date})", fontsize=12, fontweight="bold")
    plt.xlabel("Volar Momentum Score", fontsize=10)
    plt.grid(axis="x", linestyle=":", alpha=0.5)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    plt.close("all")
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode("utf-8")
    chart_html = f'<div style="text-align: center; margin-top: 15px;"><img src="data:image/png;base64,{img_b64}" style="max-width: 100%; height: auto; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);" /></div>'

# Single, Deduped Automatic Browser Download
dl_notice = ""
excel_path = top_df.attrs.get("excel_path")
if excel_path and Path(excel_path).exists():
    dl_notice = f"✓ Excel Tearsheet Ready: {Path(excel_path).name}"
    if auto_download_excel:
        try:
            dl_ok = trigger_colab_download(excel_path, cooldown_seconds=20.0)
            dl_notice += " (Downloaded)" if dl_ok else " (Ready on disk)"
        except Exception as e_dl:
            dl_notice += f" (Note: {e_dl})"

# Render collapsible output (Hidden by default, reveals on click)
full_log_text = (
    f"Market State: Latest Expected EOD: {expected_session} | Database Latest: {db_latest}\n"
    f"Configuration: Top {top_n} | Excel: {generate_excel} | Telegram: {send_telegram} | Auto-DL: {auto_download_excel}\n\n"
    + tearsheet_text
    + (("\n" + overlay_text) if overlay_text else "")
    + "\n"
    + audit_text
    + (f"\n\n{dl_notice}" if dl_notice else "")
)

display(HTML(f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 10px 0;">
  <div style="background: #1e1e2e; border: 1px solid #45475a; border-radius: 8px; padding: 12px 16px; margin-bottom: 8px; display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: 8px;">
    <div style="color: #a6e3a1; font-weight: bold; font-size: 14px;">
      ✅ Scan Complete: <span style="color: #cdd6f4;">{universe}</span> • {len(top_df) if top_df is not None else 0} Candidates Ranked • {scanned_date}
    </div>
    <div style="font-size: 12px; color: #89b4fa;">
      {dl_notice}
    </div>
  </div>

  <details style="background: #11111b; border: 1px solid #313244; border-radius: 8px; padding: 12px; cursor: pointer;">
    <summary style="font-weight: bold; font-size: 14px; color: #00adb5; padding: 4px 8px; outline: none; user-select: none;">
      ▶ 📊 Click to Show Detailed Tearsheet, Sector Matrix & Charts
    </summary>
    <div style="margin-top: 12px; cursor: default;">
      {chart_html}
      <pre style="background: #181825; border: 1px solid #313244; border-radius: 6px; padding: 12px; font-family: 'Fira Code', Menlo, Consolas, monospace; font-size: 12px; color: #cdd6f4; white-space: pre-wrap; word-break: break-word; overflow-x: auto; margin-top: 15px;">{full_log_text}</pre>
    </div>
  </details>
</div>
"""))

## ⚡ Cell 3: High-Performance C99 SIMD Backtesting & Brute Force Cockpit
**Interactive Dashboard (Run Cell First, Then Select):** Execute custom-date multi-factor backtests or launch parallel multi-threaded brute-force parameter optimization sweeps in C99 SIMD (AVX2 / NEON). Features interactive dropdowns, sliders, checkboxes, tearsheets, equity curves, and automated CSV result downloads.



In [ ]:
# ==============================================================================
# CELL 3: Multi-Architecture C99 SIMD Backtesting & Brute Force Cockpit
# ==============================================================================
import os, sys, time, inspect, io, base64
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from brute_force.bridge import compile_native_kernel
from brute_force.runner import BruteForceCampaignRunner
from scripts.scanner_hardening import trigger_colab_download

# Clean up any lingering figures from previous runs
plt.close("all")

# 1. Compile or load native C99 SIMD kernel (AVX2/NEON)
so_path = compile_native_kernel(force_recompile=False)
print(f"✓ Native SIMD Kernel active: {so_path.name} ({so_path.stat().st_size:,} bytes)")

# 2. Maintain singleton runner instance in RAM to reuse cached TensorCube
if "_bf_runner" not in globals() or _bf_runner is None:
    _bf_runner = BruteForceCampaignRunner()
runner = _bf_runner

# 3. Interactive UI Controls
w_engine_mode = widgets.RadioButtons(
    options=["🎯 Custom Date & Parameter Backtest", "⚡ Parallel Brute Force Optimization Sweep"],
    value="🎯 Custom Date & Parameter Backtest",
    description="Engine Mode:",
    layout={"width": "max-content"}
)

# Mode 1 Controls: Custom Date Backtest
era_presets = {
    "Full 10-Year (2016–2026)": ("2016-01-01", "2026-08-31"),
    "Post-COVID Bull (2020–2024)": ("2020-04-01", "2024-12-31"),
    "COVID Crash & Recovery (2020)": ("2020-01-01", "2020-09-30"),
    "Recent 2-Year (2024–2026)": ("2024-01-01", "2026-10-14"),
    "Mid-Cap Bear (2018–2019)": ("2018-01-01", "2019-12-31"),
    "Custom Dates": ("", ""),
}

w_bt_preset = widgets.Dropdown(options=list(era_presets.keys()), value="Full 10-Year (2016–2026)", description="Era Preset:")
w_bt_start = widgets.Text(value="2016-01-01", description="Start Date:")
w_bt_end = widgets.Text(value="2026-08-31", description="End Date:")
w_bt_universe = widgets.Dropdown(options=["NIFTY750", "NIFTY500", "ALL"], value="NIFTY750", description="Universe:")
w_bt_schedule = widgets.Dropdown(options=["monthly", "weekly", "quarterly", "half_yearly", "yearly"], value="monthly", description="Schedule:")
w_bt_top_n = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Top N:")
w_bt_lookback = widgets.Dropdown(options=[126, 189, 252, 315], value=252, description="Lookback:")
w_bt_ema = widgets.Dropdown(options=[50, 100, 150, 200, 250], value=200, description="EMA Trend:")
w_bt_retrace = widgets.FloatSlider(value=0.80, min=0.60, max=0.95, step=0.05, description="Retrace:")
w_bt_sector_cap = widgets.IntSlider(value=2, min=1, max=5, description="Max/Sector:")
w_bt_sizing = widgets.Dropdown(options=["atr_risk_parity", "equal_weight"], value="atr_risk_parity", description="Sizing Mode:")

w_alpha_res = widgets.Checkbox(value=False, description="Residual Momentum (Blitz 2011)")
w_alpha_vol = widgets.Checkbox(value=False, description="EWMA Vol Targeting (Harvey 2018)")
w_alpha_flow = widgets.Checkbox(value=False, description="Delivery Flow Absorption Veto")

w_btn_run_bt = widgets.Button(description="🚀 Run Custom Backtest", button_style="primary", icon="play")

# Mode 2 Controls: Brute Force Parameter Sweep
sweep_presets = [
    "Tier 1: Multi-Era & Multi-Schedule Sweep (35 runs)",
    "Tier 2: Systematic Filter Ablation (44 runs)",
    "Tier 3: Rapid Neighborhood Grid (810 runs)",
    "Tier 3: Deep Comprehensive Grid (10,000 runs)",
    "Tier 3: Full Factorial Master Matrix (61,440 runs)",
    "Tier 3: Mega Factorial Matrix (250,000 runs)",
    "Tier 3: Ultra Factorial Matrix (1,000,000 runs)",
    "Tier 4: Monte Carlo Walk-Forward & DSR (500 runs)",
    "Custom Parameter Sweep",
]
w_sw_preset = widgets.Dropdown(options=sweep_presets, value="Tier 1: Multi-Era & Multi-Schedule Sweep (35 runs)", description="Sweep Type:")
w_sw_universe = widgets.Dropdown(options=["NIFTY750", "NIFTY500", "ALL"], value="NIFTY750", description="Universe:")
w_sw_schedule = widgets.Dropdown(options=["monthly", "weekly", "quarterly", "half_yearly", "yearly"], value="monthly", description="Schedule:")
w_sw_start = widgets.Text(value="2016-01-01", description="Start Date:")
w_sw_end = widgets.Text(value="2026-08-31", description="End Date:")
w_sw_threads = widgets.IntSlider(value=min(8, os.cpu_count() or 4), min=1, max=16, description="Threads:")
w_sw_auto_dl = widgets.Checkbox(value=True, description="Auto-Download CSV 📥")

w_sw_alpha_res = widgets.Checkbox(value=False, description="Residual Momentum (Blitz 2011)")
w_sw_alpha_vol = widgets.Checkbox(value=False, description="EWMA Vol Targeting (Harvey 2018)")
w_sw_alpha_flow = widgets.Checkbox(value=False, description="Delivery Flow Absorption Veto")

w_btn_run_sw = widgets.Button(description="⚡ Launch Parallel C99 Sweep", button_style="warning", icon="bolt")

w_output_bt = widgets.Output()

def on_preset_changed(change):
    if change["new"] in era_presets and change["new"] != "Custom Dates":
        s, e = era_presets[change["new"]]
        w_bt_start.value = s
        w_bt_end.value = e

w_bt_preset.observe(on_preset_changed, names="value")

box_backtest = widgets.VBox([
    widgets.HBox([w_bt_preset, w_bt_universe]),
    widgets.HBox([w_bt_start, w_bt_end]),
    widgets.HBox([w_bt_schedule, w_bt_top_n]),
    widgets.HBox([w_bt_lookback, w_bt_ema]),
    widgets.HBox([w_bt_retrace, w_bt_sector_cap]),
    widgets.HBox([w_bt_sizing]),
    widgets.HBox([w_alpha_res, w_alpha_vol, w_alpha_flow]),
    w_btn_run_bt
])

box_sweep = widgets.VBox([
    widgets.HBox([w_sw_preset, w_sw_universe]),
    widgets.HBox([w_sw_schedule, w_sw_threads]),
    widgets.HBox([w_sw_start, w_sw_end]),
    widgets.HBox([w_sw_alpha_res, w_sw_alpha_vol, w_sw_alpha_flow]),
    widgets.HBox([w_sw_auto_dl]),
    w_btn_run_sw
])

ui_container = widgets.VBox([box_backtest])

def on_mode_changed(change):
    if "Custom" in change["new"]:
        ui_container.children = [box_backtest]
    else:
        ui_container.children = [box_sweep]

w_engine_mode.observe(on_mode_changed, names="value")

_bt_is_running = False

def handle_run_backtest(b):
    global _bt_is_running
    if _bt_is_running:
        return
    _bt_is_running = True
    w_btn_run_bt.disabled = True
    _orig_bt_desc = w_btn_run_bt.description
    w_btn_run_bt.description = "⏳ Simulating..."
    with w_output_bt:
        clear_output()
        print(f"🚀 Initializing C99 Custom Backtest [{w_bt_universe.value}]...")
        print(f"  • Date Span:  {w_bt_start.value} ➔ {w_bt_end.value} ({w_bt_schedule.value})")
        print(f"  • Parameters: Lookback={w_bt_lookback.value}d | EMA={w_bt_ema.value}d | Top {w_bt_top_n.value} | Retrace={w_bt_retrace.value:.2f} | Max/Sec={w_bt_sector_cap.value}")
        print(f"  • Sizing:     {w_bt_sizing.value} | Alphas: ResMom={w_alpha_res.value}, VolTarget={w_alpha_vol.value}, Delivery={w_alpha_flow.value}")
        t0 = time.perf_counter()
        try:
            res = runner.run_custom_backtest(
                start_date=w_bt_start.value,
                end_date=w_bt_end.value or None,
                universe_name=w_bt_universe.value,
                schedule=w_bt_schedule.value,
                lookback=w_bt_lookback.value,
                ema_trend=w_bt_ema.value,
                top_n=w_bt_top_n.value,
                retrace_threshold=w_bt_retrace.value,
                max_per_sector=w_bt_sector_cap.value,
                sizing_mode=w_bt_sizing.value,
                use_residual_mom=w_alpha_res.value,
                use_vol_targeting=w_alpha_vol.value,
                use_delivery_veto=w_alpha_flow.value
            )
            elapsed_ms = (time.perf_counter() - t0) * 1000.0

            tearsheet_text = (
                "=" * 76 + "\n"
                f" PROJECT MIP: INSTITUTIONAL TEARSHEET ({res.get('start_date', w_bt_start.value)} ➔ {res.get('end_date', w_bt_end.value)})\n"
                + "=" * 76 + "\n"
                f"  • Initial Capital:    ₹{res.get('initial_capital', 1_000_000.0):,.2f}\n"
                f"  • Ending Capital:     ₹{res['ending_equity']:,.2f} (Total Return: {((res['ending_equity']/max(1.0, res.get('initial_capital', 1_000_000.0)))-1)*100:+.2f}%)\n"
                f"  • Annualized CAGR:    {res['cagr_pct']:+.2f}%\n"
                f"  • Sharpe Ratio:       {res['sharpe_ratio']:.2f}\n"
                f"  • Sortino Ratio:      {res['sortino_ratio']:.2f}\n"
                f"  • Calmar Ratio:       {res['calmar_ratio']:.2f}\n"
                f"  • Maximum Drawdown:   {res['max_drawdown_pct']:.2f}%\n"
                f"  • Win Rate:           {res['win_rate_pct']:.1f}% ({res.get('winning_trades', 0)} wins / {res.get('total_trades', 0)} trades)\n"
                f"  • Annual Turnover:    {res['turnover_annual_pct']:.1f}%\n"
                f"  • C99 SIMD Latency:   {res['sim_latency_ms']:.2f} ms (Total compute: {elapsed_ms:.1f} ms)\n"
                + "=" * 76
            )

            # Plot Equity Curve & Underwater Drawdown
            plt.close("all")
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), dpi=100, gridspec_kw={"height_ratios": [2.5, 1.2]}, sharex=True)
            dates = pd.to_datetime(res["rebalance_dates"])
            eq = np.array(res["equity_curve"])
            bench = np.array(res.get("benchmark_equity", eq))
            dd = np.array(res.get("drawdown_pct_series", np.zeros_like(eq)))

            ax1.plot(dates, eq, label=f"MIP Strategy ({res['universe']}) [CAGR: {res['cagr_pct']:+.1f}%]", color="#00adb5", lw=2.2)
            ax1.plot(dates, bench, label="NIFTY 500 Benchmark", color="#ff79c6", lw=1.6, ls="--", alpha=0.85)
            ax1.set_title(f"MIP Institutional Backtest • {res['universe']} ({res['start_date']} to {res['end_date']})", fontsize=13, fontweight="bold")
            ax1.set_ylabel("Portfolio Value (₹)", fontsize=11)
            ax1.grid(True, alpha=0.25, ls=":")
            ax1.legend(loc="upper left", frameon=True)
            ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"₹{x:,.0f}"))

            ax2.fill_between(dates, dd, 0, color="#ff5555", alpha=0.4, label=f"Underwater Drawdown (Max: {res['max_drawdown_pct']:.1f}%)")
            ax2.plot(dates, dd, color="#ff5555", lw=1.2)
            ax2.set_ylabel("Drawdown %", fontsize=11)
            ax2.set_xlabel("Rebalance Session Date", fontsize=11)
            ax2.grid(True, alpha=0.25, ls=":")
            ax2.legend(loc="lower left", frameon=True)
            ax2.set_ylim([min(-35.0, np.min(dd) * 1.15 if len(dd) > 0 else -35.0), 2.0])

            plt.tight_layout()
            buf = io.BytesIO()
            plt.savefig(buf, format="png", bbox_inches="tight", dpi=100)
            plt.close("all")
            buf.seek(0)
            img_b64 = base64.b64encode(buf.read()).decode("utf-8")
            chart_html = f'<div style="text-align: center; margin-top: 15px;"><img src="data:image/png;base64,{img_b64}" style="max-width: 100%; height: auto; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);" /></div>'

            # Save CSV
            out_dir = Path("reports/brute_force")
            out_dir.mkdir(parents=True, exist_ok=True)
            csv_path = out_dir / f"backtest_{w_bt_universe.value}_{w_bt_schedule.value}_{int(w_bt_lookback.value)}d.csv"
            pd.DataFrame({
                "rebalance_date": res.get("rebalance_dates", []),
                "equity": res.get("equity_curve", []),
                "benchmark_equity": res.get("benchmark_equity", []),
                "drawdown_pct": res.get("drawdown_pct_series", []),
            }).to_csv(csv_path, index=False)

            dl_notice = f"✓ Results Saved: {csv_path.name}"
            try:
                dl_ok = trigger_colab_download(csv_path, cooldown_seconds=20.0)
                dl_notice += " (Downloaded)" if dl_ok else " (Ready on disk)"
            except Exception as e_dl:
                dl_notice += f" (Note: {e_dl})"

            display(HTML(f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 10px 0;">
  <div style="background: #1e1e2e; border: 1px solid #45475a; border-radius: 8px; padding: 12px 16px; margin-bottom: 8px; display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: 8px;">
    <div style="color: #a6e3a1; font-weight: bold; font-size: 14px;">
      ✅ Backtest Complete • CAGR: <span style="color: #f9e2af;">{res['cagr_pct']:+.2f}%</span> • Sharpe: <span style="color: #89b4fa;">{res['sharpe_ratio']:.2f}</span> • Max DD: <span style="color: #f38ba8;">{res['max_drawdown_pct']:.2f}%</span> • Latency: <span style="color: #cba6f7;">{res['sim_latency_ms']:.2f}ms</span>
    </div>
    <div style="font-size: 12px; color: #89b4fa;">
      {dl_notice}
    </div>
  </div>
  <details open style="background: #11111b; border: 1px solid #313244; border-radius: 8px; padding: 12px; cursor: pointer;">
    <summary style="font-weight: bold; font-size: 14px; color: #00adb5; padding: 4px 8px; outline: none; user-select: none;">
      ▶ 📈 Detailed Tearsheet, Equity Curves & Diagnostics
    </summary>
    <div style="margin-top: 12px; cursor: default;">
      {chart_html}
      <pre style="background: #181825; border: 1px solid #313244; border-radius: 6px; padding: 12px; font-family: 'Fira Code', Menlo, Consolas, monospace; font-size: 12px; color: #cdd6f4; white-space: pre-wrap; word-break: break-word; overflow-x: auto; margin-top: 15px;">{tearsheet_text}\n\n{dl_notice}</pre>
    </div>
  </details>
</div>
"""))
        except Exception as e:
            print(f"❌ Backtest Error: {e}")
        finally:
            w_btn_run_bt.disabled = False
            w_btn_run_bt.description = _orig_bt_desc
            _bt_is_running = False

_sw_is_running = False

def handle_run_sweep(b):
    global _sw_is_running
    if _sw_is_running:
        return
    _sw_is_running = True
    w_btn_run_sw.disabled = True
    _orig_sw_desc = w_btn_run_sw.description
    w_btn_run_sw.description = "⏳ Sweeping..."
    with w_output_bt:
        clear_output()
        print(f"⚡ Launching Parallel C99 SIMD Optimization Sweep ({w_sw_universe.value})...")
        print(f"  • Sweep Selection: {w_sw_preset.value}")
        print(f"  • Date Span:       {w_sw_start.value} ➔ {w_sw_end.value} ({w_sw_schedule.value})")
        print(f"  • Concurrency:     {w_sw_threads.value} CPU Worker Threads")
        print(f"  • Alphas:          ResMom={w_sw_alpha_res.value}, VolTarget={w_sw_alpha_vol.value}, Delivery={w_sw_alpha_flow.value}")
        t0 = time.perf_counter()
        try:
            if "Tier 1" in w_sw_preset.value:
                df = runner.run_grid_preset(
                    preset="tier1",
                    universe_name=w_sw_universe.value,
                    num_threads=w_sw_threads.value,
                    use_residual_mom=w_sw_alpha_res.value,
                    use_vol_targeting=w_sw_alpha_vol.value,
                    use_delivery_veto=w_sw_alpha_flow.value
                )
            elif "Tier 2" in w_sw_preset.value:
                df = runner.run_grid_preset(
                    preset="tier2",
                    universe_name=w_sw_universe.value,
                    num_threads=w_sw_threads.value,
                    use_residual_mom=w_sw_alpha_res.value,
                    use_vol_targeting=w_sw_alpha_vol.value,
                    use_delivery_veto=w_sw_alpha_flow.value
                )
            elif "Tier 4" in w_sw_preset.value or "Monte Carlo" in w_sw_preset.value:
                df = runner.run_grid_preset(
                    preset="tier4",
                    universe_name=w_sw_universe.value,
                    num_threads=w_sw_threads.value,
                )
            elif "Ultra" in w_sw_preset.value or "1,000,000" in w_sw_preset.value:
                df = runner.run_grid_preset(
                    preset="tier3_ultra",
                    universe_name=w_sw_universe.value,
                    num_threads=w_sw_threads.value,
                    use_residual_mom=w_sw_alpha_res.value,
                    use_vol_targeting=w_sw_alpha_vol.value,
                    use_delivery_veto=w_sw_alpha_flow.value
                )
            elif "Mega" in w_sw_preset.value or "250,000" in w_sw_preset.value:
                df = runner.run_grid_preset(
                    preset="tier3_mega",
                    universe_name=w_sw_universe.value,
                    num_threads=w_sw_threads.value,
                    use_residual_mom=w_sw_alpha_res.value,
                    use_vol_targeting=w_sw_alpha_vol.value,
                    use_delivery_veto=w_sw_alpha_flow.value
                )
            elif "Full Factorial" in w_sw_preset.value or "61,440" in w_sw_preset.value:
                df = runner.run_grid_preset(
                    preset="tier3_full",
                    universe_name=w_sw_universe.value,
                    num_threads=w_sw_threads.value,
                    use_residual_mom=w_sw_alpha_res.value,
                    use_vol_targeting=w_sw_alpha_vol.value,
                    use_delivery_veto=w_sw_alpha_flow.value
                )
            elif "Tier 3: Deep" in w_sw_preset.value or "10,000" in w_sw_preset.value:
                df = runner.run_grid_preset(
                    preset="tier3_deep",
                    universe_name=w_sw_universe.value,
                    num_threads=w_sw_threads.value,
                    use_residual_mom=w_sw_alpha_res.value,
                    use_vol_targeting=w_sw_alpha_vol.value,
                    use_delivery_veto=w_sw_alpha_flow.value
                )
            elif "Tier 3" in w_sw_preset.value or "Rapid" in w_sw_preset.value:
                df = runner.run_grid_preset(
                    preset="tier3_rapid",
                    universe_name=w_sw_universe.value,
                    num_threads=w_sw_threads.value,
                    use_residual_mom=w_sw_alpha_res.value,
                    use_vol_targeting=w_sw_alpha_vol.value,
                    use_delivery_veto=w_sw_alpha_flow.value
                )
            else:
                df = runner.run_custom_sweep(
                    start_date=w_sw_start.value,
                    end_date=w_sw_end.value or None,
                    universe_name=w_sw_universe.value,
                    schedules=[w_sw_schedule.value],
                    num_threads=w_sw_threads.value,
                    use_residual_mom=w_sw_alpha_res.value,
                    use_vol_targeting=w_sw_alpha_vol.value,
                    use_delivery_veto=w_sw_alpha_flow.value
                )

            elapsed_s = time.perf_counter() - t0
            sims_per_sec = len(df) / max(0.001, elapsed_s)

            top10 = df.head(10).copy()
            cols_to_show = [c for c in ["param_id", "lookback", "ema_trend", "top_n", "retrace_threshold", "cagr", "sharpe", "sortino", "max_drawdown", "win_rate", "ending_equity"] if c in top10.columns]
            sweep_summary_text = (
                f"✓ Optimization Sweep Complete: {len(df):,} parameter sets evaluated in {elapsed_s:.2f}s ({sims_per_sec:,.0f} sims/sec)\n\n"
                "🏆 TOP 10 OPTIMAL PARAMETER CONFIGURATIONS (RANKED BY SHARPE):\n"
                + top10[cols_to_show].to_string(index=False)
            )

            out_dir = Path("reports/brute_force")
            out_dir.mkdir(parents=True, exist_ok=True)
            csv_path = out_dir / "brute_force_sweep_results.csv"
            df.to_csv(csv_path, index=False)
            dl_notice = f"✓ Sweep Dataset Saved: {csv_path.name} ({len(df)} rows)"

            plt.close("all")
            fig = plt.figure(figsize=(10, 4.5), dpi=100)
            top_plot = top10.sort_values("sharpe", ascending=True)
            labels = [f"LB={int(r['lookback'])}d EMA={int(r['ema_trend'])}d N={int(r['top_n'])}" for _, r in top_plot.iterrows()]
            plt.barh(labels, top_plot["sharpe"], color="#00adb5")
            plt.title("Top 10 Configurations — Sharpe Ratio Distribution", fontsize=12, fontweight="bold")
            plt.xlabel("Annualized Sharpe Ratio", fontsize=10)
            plt.grid(axis="x", linestyle=":", alpha=0.5)
            plt.tight_layout()

            buf = io.BytesIO()
            plt.savefig(buf, format="png", bbox_inches="tight", dpi=100)
            plt.close("all")
            buf.seek(0)
            img_b64 = base64.b64encode(buf.read()).decode("utf-8")
            chart_html = f'<div style="text-align: center; margin-top: 15px;"><img src="data:image/png;base64,{img_b64}" style="max-width: 100%; height: auto; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);" /></div>'

            if w_sw_auto_dl.value:
                try:
                    dl_ok = trigger_colab_download(csv_path, cooldown_seconds=20.0)
                    dl_notice += " (Downloaded)" if dl_ok else " (Ready on disk)"
                except Exception as e_dl:
                    dl_notice += f" (Note: {e_dl})"

            best_sharpe = top10.iloc[0]['sharpe'] if not top10.empty and 'sharpe' in top10.columns else 0.0
            best_cagr = top10.iloc[0]['cagr'] if not top10.empty and 'cagr' in top10.columns else 0.0
            display(HTML(f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 10px 0;">
  <div style="background: #1e1e2e; border: 1px solid #45475a; border-radius: 8px; padding: 12px 16px; margin-bottom: 8px; display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: 8px;">
    <div style="color: #a6e3a1; font-weight: bold; font-size: 14px;">
      ✅ Sweep Complete: <span style="color: #cdd6f4;">{len(df):,} configurations</span> • Top Sharpe: <span style="color: #89b4fa;">{best_sharpe:.2f}</span> • Top CAGR: <span style="color: #f9e2af;">{best_cagr:+.2f}%</span> • Elapsed: <span style="color: #cba6f7;">{elapsed_s:.2f}s</span>
    </div>
    <div style="font-size: 12px; color: #89b4fa;">
      {dl_notice}
    </div>
  </div>
  <details open style="background: #11111b; border: 1px solid #313244; border-radius: 8px; padding: 12px; cursor: pointer;">
    <summary style="font-weight: bold; font-size: 14px; color: #00adb5; padding: 4px 8px; outline: none; user-select: none;">
      ▶ 🏆 Top 10 Configurations & Parameter Chart
    </summary>
    <div style="margin-top: 12px; cursor: default;">
      {chart_html}
      <pre style="background: #181825; border: 1px solid #313244; border-radius: 6px; padding: 12px; font-family: 'Fira Code', Menlo, Consolas, monospace; font-size: 12px; color: #cdd6f4; white-space: pre-wrap; word-break: break-word; overflow-x: auto; margin-top: 15px;">{sweep_summary_text}\n\n{dl_notice}</pre>
    </div>
  </details>
</div>
"""))
        except Exception as e:
            print(f"❌ Sweep Error: {e}")
        finally:
            w_btn_run_sw.disabled = False
            w_btn_run_sw.description = _orig_sw_desc
            _sw_is_running = False

try:
    w_btn_run_bt.on_click(handle_run_backtest, remove=True)
    w_btn_run_sw.on_click(handle_run_sweep, remove=True)
except Exception:
    pass
w_btn_run_bt.on_click(handle_run_backtest)
w_btn_run_sw.on_click(handle_run_sweep)

display(widgets.VBox([
    w_engine_mode,
    ui_container,
    w_output_bt
]))


## 🖥️ Cell 4: Headless CLI Execution & Batch Commands
Run any of the 18 CLI functions directly from the notebook.


In [ ]:
# ==============================================================================
# CELL 4: Headless Terminal Execution Examples
# ==============================================================================
# 1. Run live momentum scan on NIFTY 750 (Total Market)
!python main.py --option 1 --universe NIFTY750 --allow-stale

# 2. Run blazing fast brute force baseline simulation
# !python main.py --brute-force --brute-force-mode baseline

# 3. Run complete 4-tier optimization matrix campaign
# !python scripts/run_optimization_matrix.py